# Prepare Calendar Table

Generates `adb_caresync.silver.calendar` — a standard date dimension table spanning **1940-01-01 to 2030-12-31** with all key calendar attributes.

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.types import DateType

# Generate date range from 1940-01-01 to 2030-12-31
df_calendar = (
    spark.sql("SELECT explode(sequence(DATE '1940-01-01', DATE '2030-12-31', INTERVAL 1 DAY)) AS date")
    .withColumn("date_key", F.date_format("date", "yyyyMMdd").cast("int"))
    .withColumn("year", F.year("date"))
    .withColumn("month", F.month("date"))
    .withColumn("day", F.dayofmonth("date"))
    .withColumn("quarter", F.quarter("date"))
    .withColumn("week_of_year", F.weekofyear("date"))
    .withColumn("day_of_week", F.dayofweek("date"))  # 1=Sunday, 7=Saturday
    .withColumn("day_of_year", F.dayofyear("date"))
    .withColumn("month_name", F.date_format("date", "MMMM"))
    .withColumn("month_short_name", F.date_format("date", "MMM"))
    .withColumn("day_name", F.date_format("date", "EEEE"))
    .withColumn("day_short_name", F.date_format("date", "EEE"))
    .withColumn("quarter_name", F.concat(F.lit("Q"), F.quarter("date")))
    .withColumn("year_month", F.date_format("date", "yyyy-MM"))
    .withColumn("year_quarter", F.concat(F.year("date"), F.lit("-Q"), F.quarter("date")))
    .withColumn("is_weekend", F.when(F.dayofweek("date").isin(1, 7), True).otherwise(False))
    .withColumn("is_weekday", F.when(F.dayofweek("date").isin(1, 7), False).otherwise(True))
    .withColumn("is_leap_year",
        F.when(
            (F.year("date") % 4 == 0) & ((F.year("date") % 100 != 0) | (F.year("date") % 400 == 0)),
            True
        ).otherwise(False)
    )
    .withColumn("first_day_of_month", F.trunc("date", "month"))
    .withColumn("last_day_of_month", F.last_day("date"))
    .withColumn("first_day_of_quarter", F.trunc("date", "quarter"))
    .withColumn("last_day_of_quarter", F.date_sub(F.add_months(F.trunc("date", "quarter"), 3), 1))
    .withColumn("first_day_of_year", F.trunc("date", "year"))
    .withColumn("last_day_of_year", F.make_date(F.year("date"), F.lit(12), F.lit(31)))
    .withColumn("days_in_month", F.dayofmonth(F.last_day("date")))
    .withColumn("month_year", F.date_format("date", "MMM-yyyy"))
    .withColumn("fiscal_year",
        F.when(F.month("date") >= 4, F.year("date") + 1).otherwise(F.year("date"))
    )  # Fiscal year starting April
    .withColumn("fiscal_quarter",
        F.when(F.quarter("date") == 2, 1)
         .when(F.quarter("date") == 3, 2)
         .when(F.quarter("date") == 4, 3)
         .otherwise(4)
    )  # Fiscal quarter (Apr-based)
)

# Reorder columns with date_key first
df_calendar = df_calendar.select(
    "date_key", "date", "year", "month", "day", "quarter", "week_of_year",
    "day_of_week", "day_of_year", "month_name", "month_short_name",
    "day_name", "day_short_name", "quarter_name", "year_month", "year_quarter",
    "month_year", "is_weekend", "is_weekday", "is_leap_year",
    "first_day_of_month", "last_day_of_month", "first_day_of_quarter",
    "last_day_of_quarter", "first_day_of_year", "last_day_of_year",
    "days_in_month", "fiscal_year", "fiscal_quarter"
)

# Write to Unity Catalog table
df_calendar.write.mode("overwrite").saveAsTable("adb_caresync.silver.calendar")

print(f"Calendar table created with {df_calendar.count()} rows")
display(spark.table("adb_caresync.silver.calendar").limit(10))